# وصّل بذكاء
## مشروع الإنذار المبكر لتأخر التوصيل
إعداد: عبدالله الصقيه

امتداد Labs 1–3. بيانات اصطناعية معلنة، نموذج تعلم آلة فعلي، ومقارنة سائق ومسار بديلين. ليس خدمة تشغيلية ولا تنفيذا مفترضا لـ Lab 4.

شغّل الخلايا بالترتيب في Google Colab. متطلبات البيئة موثقة في requirements.txt؛ المكتبات الأساسية عادة متوفرة في Colab.

## الإعداد
استيراد المكتبات وإنشاء مجلد المخرجات. عند استخدام Colab تحفظ الملفات في outputs داخل الجلسة ويمكن تنزيلها من قائمة Files.

In [1]:
"""Wassel Smart — reproducible educational model. All data are synthetic."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, accuracy_score, confusion_matrix, brier_score_loss

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(exist_ok=True)
RAW_FEATURES = ['Distance_Km', 'Driver_Orders', 'Traffic_Level']
FEATURES = RAW_FEATURES + ['Distance_Traffic', 'Orders_Traffic']
TARGET = 'Delayed'
RANDOM_STATE = 42



## 1 — البيانات الاصطناعية
نولد 3,000 سجل بضوضاء عشوائية وعلاقات افتراضية. هذه ليست بيانات حقيقية؛ خصائصها لا تثبت طبيعة التوصيل الفعلي. نفس أسماء المدخلات والهدف من Lab 1.

In [2]:
# 1. Synthetic data generation: assumptions, not observed delivery records.
# An unobserved random factor represents variation absent from the inputs.
rng = np.random.default_rng(RANDOM_STATE)
n = 3000
distance = rng.uniform(1, 50, n).round(1)
orders = rng.integers(0, 11, n)
traffic = rng.choice([1, 2, 3], n, p=[.35, .4, .25])
log_odds = (-4.8 + .065 * distance + .22 * orders + .55 * (traffic - 1)
            + .025 * distance * (traffic - 1) + .09 * orders * (traffic - 1)
            + rng.normal(0, .6, n))
probability = 1 / (1 + np.exp(-log_odds))
df = pd.DataFrame({'Distance_Km': distance, 'Driver_Orders': orders,
                   'Traffic_Level': traffic, 'Delayed': rng.binomial(1, probability)})
df.to_csv(OUTPUT / 'synthetic_deliveries.csv', index=False)
print('Synthetic records:', len(df), '| Delayed prevalence:', round(df[TARGET].mean(), 3))



Synthetic records: 3000 | Delayed prevalence: 0.396


## 2 — التقسيم وهندسة الخصائص
20% للاختبار، والباقي للتدريب. تفاعلات المسافة والطلبات مع الازدحام مأخوذة من Lab 2. تبقى المعالجة داخل Pipeline لتتعلم من طية التدريب فقط.

In [3]:
# 2. Keep 20% untouched during training, preprocessing, and model tuning.
X_train, X_test, y_train, y_test = train_test_split(
    df[RAW_FEATURES], df[TARGET], test_size=.2, stratify=df[TARGET], random_state=RANDOM_STATE)

def add_delivery_features(data):
    result = data[RAW_FEATURES].copy()
    result['Distance_Traffic'] = result.Distance_Km * result.Traffic_Level
    result['Orders_Traffic'] = result.Driver_Orders * result.Traffic_Level
    return result

def pipeline(classifier):
    return Pipeline([('features', FunctionTransformer(add_delivery_features)),
                     ('imputer', SimpleImputer(strategy='median')),
                     ('scaler', StandardScaler()), ('classifier', classifier)])



## 3 — النماذج وضبط المعلمات
مقارنة Logistic Regression وRandom Forest وGradient Boosting. البحث في ثمانية إعدادات Gradient Boosting بتحقق طبقي من ثلاث طيات تدريب، وفق مقياس AP. لا تستعمل مجموعة الاختبار أثناء الضبط.

In [4]:
# 3. Baselines and the Lab 3 gradient-boosting search, training folds only.
models = {
    'Logistic Regression': pipeline(LogisticRegression(max_iter=1000, random_state=42)),
    'Random Forest': pipeline(RandomForestClassifier(n_estimators=120, max_depth=6, min_samples_leaf=8, random_state=42)),
    'Default Gradient Boosting': pipeline(GradientBoostingClassifier(random_state=42))
}
for model in models.values():
    model.fit(X_train, y_train)
search = GridSearchCV(pipeline(GradientBoostingClassifier(random_state=42)),
    {'classifier__n_estimators':[60,120], 'classifier__learning_rate':[.04,.08],
     'classifier__max_depth':[1,2]}, scoring='average_precision',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42), error_score='raise')
search.fit(X_train, y_train)
best_model = search.best_estimator_
models['Tuned Gradient Boosting'] = best_model
print('Best training CV AP:', round(search.best_score_, 4), '| Parameters:', search.best_params_)



Best training CV AP: 0.8268 | Parameters: {'classifier__learning_rate': 0.04, 'classifier__max_depth': 2, 'classifier__n_estimators': 120}


## 4 — التقييم
نستخدم حد تصنيف ثابت 0.5. AP هو Average Precision وليس دقة التصنيف Accuracy. تبين المقارنة أن النموذج الأكثر تعقيدا ليس بالضرورة الأفضل. احتفظنا بالنموذج المضبوط في Lab 3 للتطبيق، ولم نختر الفائز بناء على الاختبار.

In [5]:
# 4. Holdout evaluation once; fixed 0.5 threshold, not optimized on test labels.
# The deployed model is the CV-selected Gradient Boosting model, not a test-set winner.
rows=[]
for name, model in models.items():
    p=model.predict_proba(X_test)[:,1]
    pred=(p >= .5).astype(int)
    rows.append({'model':name, 'AP':average_precision_score(y_test,p),
        'ROC_AUC':roc_auc_score(y_test,p), 'F1':f1_score(y_test,pred),
        'accuracy':accuracy_score(y_test,pred), 'Brier':brier_score_loss(y_test,p)})
print(pd.DataFrame(rows).round(4).to_string(index=False))
p=best_model.predict_proba(X_test)[:,1]
metrics={'data_source':'synthetic', 'records':n, 'train_records':len(X_train),
    'test_records':len(X_test), 'prevalence':float(y_test.mean()), 'seed':42,
    'threshold':.5, 'cv_ap':float(search.best_score_), 'best_params':search.best_params_,
    'models':rows, 'confusion_matrix':confusion_matrix(y_test,p>=.5).tolist(),
    'sklearn_version':sklearn.__version__}
(OUTPUT/'metrics.json').write_text(json.dumps(metrics,indent=2))
pd.DataFrame(rows).to_csv(OUTPUT/'model_comparison.csv',index=False)



                    model     AP  ROC_AUC     F1  accuracy  Brier
      Logistic Regression 0.8640   0.8909 0.7429    0.8050 0.1285
            Random Forest 0.8545   0.8864 0.7555    0.8133 0.1315
Default Gradient Boosting 0.8520   0.8818 0.7549    0.8117 0.1328
  Tuned Gradient Boosting 0.8517   0.8832 0.7473    0.8050 0.1325


## 5 — تصدير النموذج
تُصدّر الأشجار الحقيقية إلى JSON ليستخدمها المتصفح. اختبار مستقل يقارن 200 تنبؤ مع scikit-learn. لا تُستبدل تنبؤات النموذج بقواعد يدوية.

In [6]:
# 5. Export actual trained trees for identical inference in the browser.
clf=best_model.named_steps['classifier']
scaler=best_model.named_steps['scaler']
prior=clf.init_.class_prior_[1]
trees=[]
for est in clf.estimators_[:,0]:
    t=est.tree_
    trees.append({'left':t.children_left.tolist(),'right':t.children_right.tolist(),
                  'feature':t.feature.tolist(),'threshold':t.threshold.tolist(),
                  'value':t.value[:,0,0].tolist()})
export={'features':FEATURES,'mean':scaler.mean_.tolist(),'scale':scaler.scale_.tolist(),
    'initial_log_odds':float(np.log(prior/(1-prior))), 'learning_rate':clf.learning_rate,
    'trees':trees,'importance':clf.feature_importances_.tolist()}
(OUTPUT/'model.json').write_text(json.dumps(export,separators=(',',':')))
# Fixtures validate the exported evaluator against scikit-learn, not hand-coded scores.
fixture_x=X_test.iloc[:200]
(OUTPUT/'parity_fixtures.json').write_text(json.dumps({'inputs':fixture_x.to_dict('records'),
    'probabilities':best_model.predict_proba(fixture_x)[:,1].tolist()}))
print('Exported model and evaluation artifacts to', OUTPUT)


Exported model and evaluation artifacts to /workspace/sites/wassel-smart/project/outputs


## 6 — مقارنة سيناريوهات
نغير عدد طلبات السائق في سيناريو السائق البديل، أو المسافة والازدحام في سيناريو المسار. هذه مقارنات افتراضية لا تثبت أثر التدخل سببيا. لا نستخدم اسم مدينة أو هوية سائق لم يقدمها المستخدم.

In [7]:
scenarios = pd.DataFrame({
    "Scenario": ["Current", "Alternative driver", "Alternative route"],
    "Distance_Km": [25, 25, 30],
    "Driver_Orders": [7, 2, 7],
    "Traffic_Level": [3, 3, 2]
})
scenarios["Delay_Probability"] = best_model.predict_proba(scenarios[RAW_FEATURES])[:, 1]
scenarios["Reduction_Percentage_Points"] = (scenarios.Delay_Probability.iloc[0] - scenarios.Delay_Probability) * 100
print(scenarios.round(4).to_string(index=False))
scenarios.to_csv(OUTPUT / "example_scenarios.csv", index=False)


          Scenario  Distance_Km  Driver_Orders  Traffic_Level  Delay_Probability  Reduction_Percentage_Points
           Current           25              7              3             0.8137                       0.0000
Alternative driver           25              2              3             0.4189                      39.4792
 Alternative route           30              7              2             0.6071                      20.6642


## الخلاصة
لا توجد دقة 100% مصطنعة: إدخال الضوضاء وحجم بيانات أكبر يتيحان تقييما تعليميا أكثر واقعية. جميع المقاييس على بيانات اصطناعية من التوزيع نفسه؛ يلزم جمع بيانات تشغيل حقيقية، واختبار زمني مستقل، ومعايرة الاحتمالات قبل أي استخدام فعلي. أهمية الخصائص ليست دليلا سببيا. متطلبات Lab 4 تبقى بانتظار تعليمات المدربة.